# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json), describing clinicopathological and molecular data for cancer survivors with second primary colorectal cancer.

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and their fields, referencing all with their `@id`.

In [ ]:
# List all record sets in the dataset and summarize available fields in each
record_sets = [r for r in dataset.metadata.record_set]
if not record_sets:
    print('No record sets defined explicitly in @recordSet. Attempting to infer from distribution...')
    # mlcroissant often auto-infers record sets from distributions if not given
    record_sets = [r['@id'] for r in dataset.to_json()['distribution']]

record_set_overview = []
for rs in dataset.record_sets:
    rs_id = rs['@id']
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        # sometimes single field returned directly
        fields = [fields]
    field_ids = [f['@id'] for f in fields]
    record_set_overview.append({
        'record_set_id': rs_id,
        'field_ids': field_ids
    })
    print(f"RecordSet @id: {rs_id}")
    print(f"  Fields (@id): {field_ids if field_ids else '[Not defined at this level]'}\n")
if not record_set_overview:
    print('No explicit record sets found. You may need to inspect available distributions or columns.')

## 3. Data Extraction
Load data from a record set into a DataFrame for analysis. Record set and field `@id`s are used for extraction.

In [ ]:
# For this dataset, record sets are inferred from the distribution objects.
# We'll extract the @id's from the croissant schema distributions for tabular data.

schema_json = dataset.to_json()
distribution_ids = [d['@id'] for d in schema_json.get('distribution', []) if 'DataFileObject' in d.get('@type', '') or d.get('@type') == 'dv:DataFileObject']
if not distribution_ids:
    # fallback: use all distribution objects
    distribution_ids = [d['@id'] for d in schema_json.get('distribution', [])]

print('Available Data Record Sets (@id):')
for i, ds_id in enumerate(distribution_ids):
    print(f"{i+1}. {ds_id}")

# Load each record set (identified by distribution @id as per croissant)
dataframes = {}
for record_set_id in distribution_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame from RecordSet @id: {record_set_id} | {df.shape[0]} rows × {df.shape[1]} cols")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Preview what's in the first record set DataFrame
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nColumn @ids for main record set ({main_rs_id}):")
    print(list(dataframes[main_rs_id].columns))
    dataframes[main_rs_id].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Try filtering and processing the data using key column `@id`s. We'll demonstrate with key numeric and categorical fields, referencing columns by their `@id`.

In [ ]:
# Adjust these @id's based on the real dataset's column names, discovered above.

# We'll use the main record set as loaded above
main_rs_id = list(dataframes.keys())[0] if dataframes else None

if main_rs_id:
    df = dataframes[main_rs_id]
    print(f"Columns in main DataFrame ({main_rs_id}):\n{df.columns.tolist()}")

    # Attempt to select a numeric field (guess using pandas select_dtypes)
    numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if not numeric_cols:
        # try to cast any likely numeric columns (often medical ids contain Age, etc.)
        numeric_candidates = [c for c in df.columns if 'age' in c.lower() or 'interval' in c.lower()]
        for c in numeric_candidates:
            try:
                df[c] = pd.to_numeric(df[c])
            except Exception:
                continue
        numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()

    print(f"Numeric columns found: {numeric_cols}")
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        # Filter rows where the value is above the mean (as an example)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head(3))

        # Normalize the field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Select a grouping field (categorical)
        # Try to use one with few unique values
        candidate_group_fields = [c for c in df.columns if 1 < df[c].nunique() < 8]
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df)
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric columns detected for EDA.")
else:
    print("Main record set not available for EDA.")

## 5. Visualization
Visualize data distributions and relationships using the fields referenced by their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

if main_rs_id and len(numeric_cols) >= 1:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    # Boxplot by group_field, if available
    if 'group_field_id' in locals() and df[group_field_id].nunique() >= 2:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

We have demonstrated the use of the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) via its Croissant schema with `mlcroissant` for structured data exploration. Using only schema `@id` references, we loaded the package metadata and tabular records, identified key fields, ran descriptive analysis, and visualized data distributions. This approach ensures transparent and reproducible biomedical analysis using FAIR medical data.